# **Generating adversarial noise examples using FGSM**
By Charlot Eberlein, Loughborough University • 11.01.26 • 
[LinkedIn](https://www.linkedin.com/in/charloteberlein/) • 
[Portfolio](https://charloteberlein.github.io) • 
[GitHub](https://github.com/charloteberlein)

In [ ]:
%%capture
import tensorflow as tf
import keras as k
import matplotlib.pyplot as plt
import numpy as np

### **Step 1:** Load the MNIST dataset

In [ ]:
(train_im, train_l), (test_im, test_l) = k.datasets.mnist.load_data()

train_im = train_im.reshape((60000,28,28,1)).astype("float32")/255
test_im = test_im.reshape((10000,28,28,1)).astype("float32")/255

Let's quickly make sure it works using matplotlib:

In [ ]:
SIZE = 6; assert SIZE > 0
fig, ax = plt.subplots(SIZE, SIZE)
fig.suptitle(f"First {SIZE**2} MNIST digits")
for i in range(SIZE):
    for j in range(SIZE):
        p = SIZE*i + j
        ax[i][j].imshow(np.reshape(train_im[p], (28,28)), cmap="gray")
        ax[i][j].set_title(train_l[p], y=-0.1, x=0.9, color="white")
        ax[i][j].set_axis_off()
plt.tight_layout()
plt.show()

If the dataset has been loaded correctly, the result should look something like this:

![Grid of digits](resources/first_36_mnist.png)

Great. Now, let's move on to the next step.

### **Step 2:** Build a Convolutional Neural Network (CNN) with TensorFlow
The architecture we're going to use is as follows:
- **Input layer**
- Data augmentation layer (random elastic transform)
- **Convolutional layer** (ReLU activation)
- **Max-pooling layer**
- Batch normalisation
- **Convolutional layer** (ReLU activation)
- **Max-pooling layer**
- Batch normalisation
- **Fully-connected layer**
- **Output layer** (Softmax activation)

**Random elastic transform** is a data augmentation method I think is best described like the "liquify" tool in photoshop. Here, we have an intensity of 0.1 which is randomly applied in 5% of the training data.

**Batch normalisation** normalises layer inputs to a mean of 0 and standard deviation of 1, which improves training speed and stability.

In [ ]:
augmented = 0
model = k.Sequential()
model.add(k.Input(shape=(28,28,1)))
model.add(k.layers.RandomElasticTransform(factor=0.05, scale=0.1))
model.add(k.layers.Conv2D(32, activation="relu", padding="same", kernel_size=5))
model.add(k.layers.MaxPool2D(pool_size=(2,2), strides=4))
model.add(k.layers.BatchNormalization())
model.add(k.layers.Conv2D(64, activation="relu", padding="same", kernel_size=5))
model.add(k.layers.MaxPool2D(pool_size=(2,2), strides=4))
model.add(k.layers.BatchNormalization())
model.add(k.layers.Dense(128, activation="relu"))
model.add(k.layers.Flatten())
model.add(k.layers.Dense(10, activation="softmax"))
# compile the finished model
model.compile(optimizer="sgd", loss="categorical_crossentropy",
              metrics=["accuracy"])

We're also going to make another version without all the data augmentation, just to compare.

In [ ]:
augmented = 1
model = k.Sequential()
model.add(k.Input(shape=(28,28,1)))
model.add(k.layers.Conv2D(32, activation="relu", padding="same", kernel_size=5))
model.add(k.layers.MaxPool2D(pool_size=(2,2), strides=4))
model.add(k.layers.Conv2D(64, activation="relu", padding="same", kernel_size=5))
model.add(k.layers.MaxPool2D(pool_size=(2,2), strides=4))
model.add(k.layers.Dense(128, activation="relu"))
model.add(k.layers.Flatten())
model.add(k.layers.Dense(10, activation="softmax"))
# compile the finished model
model.compile(optimizer="sgd", loss="categorical_crossentropy",
              metrics=["accuracy"])

Finally, we're going to make a really simple model with just one convolutional layer. I'm going to increase the stride size here because I want this to be intentionally worse, for demonstration purposes.

In [ ]:
augmented = 2
model = k.Sequential()
model.add(k.Input(shape=(28,28,1)))
model.add(k.layers.Conv2D(32, activation="relu", padding="same", kernel_size=5))
model.add(k.layers.MaxPool2D(pool_size=(2,2), strides=8))
model.add(k.layers.Flatten())
model.add(k.layers.Dense(10, activation="softmax"))
# compile the finished model
model.compile(optimizer="sgd", loss="categorical_crossentropy",
              metrics=["accuracy"])

#### **Optional:** Train the model
Because this is a really simple problem, training should only take a minute or two. However, I kept the weights from training it earlier, so you can skip this step.

You can really see how using data augmentation paid off here, because we have a fantastic degree of accuracy:\
• **Training accuracy:** 98%\
• **Validation accuracy:** 99%

In [ ]:
# pre-process labels
train_l = k.utils.to_categorical(train_l)
test_l = k.utils.to_categorical(test_l)

result = model.fit(train_im, train_l, batch_size=64, epochs=16, verbose=1,
                   validation_data=(test_im,test_l))
if augmented == 0:
    filepath = "resources/mnist-augmented.weights.h5"
elif augmented == 1:
    filepath = "resources/mnist-norm.weights.h5"
else:
    filepath = "resources/mnist-weak.weights.h5"
model.save_weights(filepath)

fig, ax = plt.subplots()
ax.plot(result.history["loss"], color="b")
ax.plot(result.history["val_loss"], color="r")
ax.set_title("Training and validation loss")
ax.set_xlabel("Epochs")
ax.set_ylabel("Loss")
ax.set_xticks(np.arange(0,16))
ax.legend(("Training loss", "Validation loss"))
plt.show()

If you chose to train it, you'd get something like the graph below, although neural network training can be quite variable.

We use loss curves to help us identify after how many epochs we should stop training. Ideally, we want a nice smooth plateau; if the curves haven't converged yet, that means the model is underfitted, and if the loss is starting to rise again or is oscillating a lot, then that means the model is overfitted.

The distance between the curves on this graph isn't due to underfitting, though, but rather the augmentation that we added to the training data. This makes training more "difficult" for the model, resulting in a controlled increase in loss.

![Loss-Epochs curves, showing two lines in exponential decay](resources/loss_epochs_data_augmentation.png)

An ideal loss-epoch curve looks a bit more like this. The only difference between this model and the model we're using is that this one didn't have any kind of data augmentation. So, it has a nicer curve, but the model accuracy is actually lower.

![Loss-Epochs curves, showing two lines very close in exponential decay](resources/loss_epochs_curve.png)

Finally, let's do one last sanity check to make sure our model is working as it should be. We'll do this by checking a subset of the testing images ourselves.

In [ ]:
if augmented == 0:
    filepath = "resources/mnist-augmented.weights.h5"
elif augmented == 1:
    filepath = "resources/mnist-norm.weights.h5"
else:
    filepath = "resources/mnist-weak.weights.h5"

# load model and get predictions
model.load_weights(filepath)
pred = model.predict(test_im, verbose=0)

# plot
fig, ax = plt.subplots(nrows=3, ncols=5, layout="tight", figsize=(8,3),
                       gridspec_kw={"height_ratios":[1,0.1,1]})
for i in range(5):
    ax[0,i].imshow(test_im[i], cmap="gray")
    ax[0,i].set_axis_off()
for i in range(5):
    ax[1,i].text(0.5, 0.5, f"Prediction: {np.argmax(pred[i])}",
                 horizontalalignment="center", verticalalignment="center")
    ax[1,i].set_axis_off()
for i in range(5):
    ax[2,i].step(np.arange(0,10), pred[i], c="blue")
plt.show()

If everything is working correctly, you should see the following plot:

![Plot of digits and predictions](resources/mnist_eval.png)

Great! Now, we're ready to get to the fun part: FGSM!

### **Step 3:** Use FGSM to generate Adversarial Noise Examples
We're going to use some tools from TensorFlow to help us.

In [ ]:
epsilon = 0.3
input_i, target_i = 9, 13

#import tensorflow as tf
def generate_adversarial_noise(im, target):
    with tf.GradientTape() as tape:
        tape.watch(im)
        pred = model(im) # forward
        target = tf.reshape(target, pred.shape)
        loss = -k.losses.categorical_crossentropy(target, pred)
    return epsilon * tf.sign(tape.gradient(loss,im))

# np -> tf
im_tensor = tf.convert_to_tensor(test_im[input_i], dtype=tf.float32)
im_tensor = tf.reshape(im_tensor, [1,28,28,1])
target_tensor = tf.one_hot(test_l[target_i], depth=10)
perturbations = generate_adversarial_noise(im_tensor, target_tensor)[0]

pred_norm = model.predict(test_im, verbose=0)
adversarial_example = test_im+perturbations
adversarial_example = tf.clip_by_value(adversarial_example, 0, 1)
pred_adversarial = model.predict(adversarial_example, verbose=0)

# plot graph
fig, ax = plt.subplots(nrows=2, ncols=4, figsize=(8,2),
                       gridspec_kw={"height_ratios":[1,0.1]})

ax[0,0].imshow(test_im[input_i], cmap="gray_r")
ax[0,1].imshow(test_im[target_i], cmap="gray_r")
ax[0,2].imshow(perturbations, cmap="gray_r")
ax[0,3].imshow(adversarial_example[input_i], cmap="gray_r")

ax[1,0].text(0.5, 0.5, f"Prediction: {np.argmax(pred_norm[input_i])}",
             horizontalalignment="center", verticalalignment="center")
ax[1,1].text(0.5, 0.5, f"Target class: {test_l[target_i]}",
             horizontalalignment="center", verticalalignment="center")
ax[1,2].text(0.5, 0.5, "FGSM Pattern",
             horizontalalignment="center", verticalalignment="center")
ax[1,3].text(0.5, 0.5, f"Prediction: {np.argmax(pred_adversarial[input_i])}",
             horizontalalignment="center", verticalalignment="center")

for ax0 in ax:
    for ax1 in ax0:
        ax1.set_axis_off()

plt.show()